# Impact Analysis of GoodThought NGO Initiatives

![ngo_project_image](ngo_project_image.jpg)

GoodThought NGO has been a catalyst for positive change, focusing its efforts on education, healthcare, and sustainable development to make a significant difference in communities worldwide. With this mission, GoodThought has orchestrated an array of assignments aimed at uplifting underprivileged populations and fostering long-term growth.

This project offers a hands-on opportunity to explore how data-driven insights can direct and enhance these humanitarian efforts. In this project, you'll engage with the GoodThought PostgreSQL database, which encapsulates detailed records of assignments, funding, impacts, and donor activities from 2010 to 2023. This comprehensive dataset includes:

- **`Assignments`:** Details about each project, including its name, duration (start and end dates), budget, geographical region, and the impact score.
- **`Donations`:** Records of financial contributions, linked to specific donors and assignments, highlighting how financial support is allocated and utilized.
- **`Donors`:** Information on individuals and organizations that fund GoodThought’s projects, including donor types.

Refer to the below ERD diagram for a visual representation of the relationships between these data tables:
<img src="erd.png" alt="ERD" width="50%" height="50%">


You will execute SQL queries to answer two questions, as listed in the instructions. Good luck!


In [11]:
/* 
- List the top five assignments based on total value of donations, categorized by donor type. 
- The output should include four columns: 1) assignment_name, 2) region, 3) rounded_total_donation_amount rounded to two decimal places, and 4) donor_type, sorted by rounded_total_donation_amount in descending order. 
*/ 

WITH donationAmountByType AS(
	SELECT public.donations.assignment_id, 
		   public.donors.donor_type, 
	       ROUND(SUM(public.donations.amount), 2) AS rounded_total_donation_amount
	FROM public.donations
	INNER JOIN public.donors
	ON public.donations.donor_id = public.donors.donor_id
	GROUP BY public.donations.assignment_id, public.donors.donor_type
)

SELECT public.assignments.assignment_name, 
	   public.assignments.region, 
	   donationAmountByType.rounded_total_donation_amount, 
	   donationAmountByType.donor_type
FROM donationAmountByType
INNER JOIN public.assignments
ON public.assignments.assignment_id = donationAmountByType.assignment_id
ORDER BY rounded_total_donation_amount DESC
LIMIT 5;

,assignment_name,region,rounded_total_donation_amount,donor_type
0,Assignment_3033,East,3840.66,Individual
1,Assignment_300,West,3133.98,Organization
2,Assignment_4114,North,2778.57,Organization
3,Assignment_1765,West,2626.98,Organization
4,Assignment_268,East,2488.69,Individual


In [10]:
/* 
- Identify the assignment with the highest impact score in each region, ensuring that each listed assignment has received at least one donation. 
- The output should include four columns: 1) assignment_name, 2) region, 3) impact_score, and 4) num_total_donations, sorted by region in ascending order. Include only the highest-scoring assignment per region, avoiding duplicates within the same region. 
*/

WITH rankingAssignment AS(
	SELECT asm.assignment_name, 
	       asm.assignment_id, 
		   asm.region, 
		   asm.impact_score, 
		   ROW_NUMBER() OVER(
				PARTITION BY asm.region
		        ORDER BY asm.impact_score DESC) AS region_rank
	FROM public.assignments AS asm
),

donationCountByAssignmentId AS(
	SELECT don.assignment_id, 
	       COUNT(don.donation_id) AS num_total_donations  
	FROM public.donations AS don
	GROUP BY don.assignment_id
	HAVING COUNT(don.donation_id) >=1
)

SELECT rankingAssignment.assignment_name, 
	   rankingAssignment.region, 
	   rankingAssignment.impact_score, 
	   donationCountByAssignmentId.num_total_donations
FROM rankingAssignment
LEFT JOIN donationCountByAssignmentId
ON rankingAssignment.assignment_id = donationCountByAssignmentId.assignment_id
WHERE rankingAssignment.region_rank = 1
ORDER BY rankingAssignment.region ASC;


,assignment_name,region,impact_score,num_total_donations
0,Assignment_316,East,10.00,2
1,Assignment_2253,North,9.99,1
2,Assignment_3547,South,10.00,1
3,Assignment_3764,West,9.99,1
